In [1]:
!pip install -q "dacite>=1.9,<2" --upgrade

# Install your NLP packages, but force dagshub to skip checking dependencies
!pip install -q transformers datasets evaluate mlflow huggingface_hub accelerate sacrebleu
!pip install -q dagshub --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Access Kaggle Secrets
user_secrets = UserSecretsClient()

# Get the Hugging Face token
hf_token = user_secrets.get_secret("HF_TOKEN")

# Authorize Hugging Face
login(token=hf_token)


In [4]:
import mlflow

# Access Kaggle Secrets
user_secrets = UserSecretsClient()

# Pass Dagshub credentials from Kaggle Secrets to Environment Variables
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("MLFLOW_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("MLFLOW_PASSWORD")

# Connect to MLflow server and select the experiment
mlflow.set_tracking_uri("https://dagshub.com/witch2256/pomello_donatello.mlflow")
mlflow.set_experiment("deep-past-initiative-machine-translation")

# Check connection with a test run
with mlflow.start_run(run_name="Participant_3_Setup_Test"):
    mlflow.set_tag("author", "Participant 3")
    mlflow.log_param("architecture", "mBART / mT5 / Subword Seq2Seq")
    mlflow.log_metric("status", 1.0)
    print("Successful MLflow-Dagshub connection!")


Successful MLflow-Dagshub connection!
🏃 View run Participant_3_Setup_Test at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0/runs/65408d35b00d4ccb94324ac2a9afc953
🧪 View experiment at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# 1. Загрузка данных
df = pd.read_csv("/kaggle/input/datasets/alexrilenko/clean-akkadian-to-english-dataset/train_clean.csv")

# Удаляем возможные пропуска в чистых колонках
df = df.dropna(subset=["transliteration_clean", "translation_clean"]).reset_index(drop=True)

# 2. Разделение на Train и Validation (90/10)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Переводим в Hugging Face Dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Задаем языковые токены (так как Аккадский отсутствует в стандартном mBART,
# берем служебный язык для связки источников и таргета)
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "en_XX"

MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    inputs = examples["transliteration_clean"]
    targets = examples["translation_clean"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Применяем токенизацию ко всему датасету
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# Инициализируем модель
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Map:   0%|          | 0/1404 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [7]:
import evaluate
import numpy as np

chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Заменяем -100 в метках для правильного декодирования
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Небольшая очистка пробелов
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    chrf_res = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels, word_order=2) # word_order=2 дает chrF++
    bleu_res = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "chrf_pp": round(chrf_res["score"], 4),
        "bleu": round(bleu_res["bleu"] * 100, 4)
    }

In [8]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

HF_REPO_NAME = "AkkadianVoicesCom/Akkadika"
RUN_NAME = "mbart50-clean-finetuning"

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_mbart",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    
    # Reduced batch size per GPU to prevent activation spikes
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    
    gradient_checkpointing=True,
    
    optim="adafactor",
    
    generation_max_length=128,
    generation_num_beams=1,  # Greedy search during eval saves VRAM over beam search
    
    weight_decay=0.01,
    save_total_limit=1,  # Keep only best checkpoint to save disk space
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="chrf_pp",
    greater_is_better=True,
    report_to="none"
)

model.gradient_checkpointing_enable()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


# Обучение с логированием в MLflow
with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tag("author", "Participant 3")
    mlflow.set_tag("model_architecture", "mbart50")
    mlflow.set_tag("hf_repo", HF_REPO_NAME)

    mlflow.log_params({
        "model_name": MODEL_NAME,
        "lr": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "epochs": training_args.num_train_epochs,
        "max_src_len": MAX_SOURCE_LENGTH,
        "max_tgt_len": MAX_TARGET_LENGTH,
        "data_columns": "clean"
    })

    trainer.train()

    # Оценка лучшей модели на валидации
    eval_metrics = trainer.evaluate()
    mlflow.log_metrics({
        "val_chrF_pp": eval_metrics["eval_chrf_pp"],
        "val_bleu": eval_metrics["eval_bleu"],
        "val_loss": eval_metrics["eval_loss"]
    })

    # Публикация обученной модели на Hugging Face Hub
    model.push_to_hub(HF_REPO_NAME, private=True)
    tokenizer.push_to_hub(HF_REPO_NAME, private=True)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Chrf Pp,Bleu
1,39.127446,1.623794,30.230300,11.195800
2,20.732136,1.364223,35.420400,15.015700
3,15.830756,1.299343,38.505000,17.654600
4,13.064319,1.282264,39.608800,18.669700
5,11.128916,1.298088,40.205600,19.359100


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


🏃 View run mbart50-clean-finetuning at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0/runs/14caeea393ac4788bd827db50a2f3aaa
🧪 View experiment at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0


In [9]:
# Генерация предсказаний на валидационном датасете
raw_predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.where(raw_predictions.predictions != -100, raw_predictions.predictions, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

# Формирование таблицы согласно Definition of Done
val_df_results = pd.DataFrame({
    "id": val_df["oare_id"].values,
    "source_text": val_df["transliteration_clean"].values,
    "target_text": val_df["translation_clean"].values,
    "pred_translation": [p.strip() for p in decoded_preds]
})

val_df_results.to_csv("preds_val_mbart50.csv", index=False)
print("Файл предсказаний preds_val_mbart50.csv успешно сохранен!")

Файл предсказаний preds_val_mbart50.csv успешно сохранен!
